# Semantic segmentation (FCN, U-Net) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def conv2d(x,k,pad=0,stride=1,dilation=1):
    x=np.asarray(x,float); k=np.asarray(k,float)
    if pad: x=np.pad(x,pad)
    kh,kw=k.shape; dh=(kh-1)*dilation+1; dw=(kw-1)*dilation+1; out=[]
    for i in range(0,x.shape[0]-dh+1,stride):
        row=[]
        for j in range(0,x.shape[1]-dw+1,stride): row.append(float(np.sum(x[i:i+dh:dilation,j:j+dw:dilation]*k)))
        out.append(row)
    return np.array(out)
def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    ix1,iy1=max(ax1,bx1),max(ay1,by1); ix2,iy2=min(ax2,bx2),min(ay2,by2)
    inter=max(0,ix2-ix1)*max(0,iy2-iy1); union=(ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
    return inter/union if union else 0
def softmax(z):
    z=np.asarray(z,float); e=np.exp(z-z.max()); return e/e.sum()
def show(a,title,cmap='viridis'):
    plt.figure(figsize=(4,3)); plt.imshow(a,cmap=cmap); plt.colorbar(); plt.title(title)
print('setup ok')

## Semantic segmentation (FCN, U-Net)
Tiny CPU-only synthetic arrays for Semantic segmentation (FCN, U-Net): no downloads, no pretrained models, and every cell ends with an assert.

In [ ]:
# 1) a segmentation mask is a label per pixel
mask=np.array([[0,0,1],[0,1,1],[2,2,1]]); show(mask,'pixel labels')
assert len(np.unique(mask))==3

In [ ]:
# 2) pixel accuracy counts exact matches
pred=np.array([[0,0,1],[0,0,1],[2,2,1]]); gt=np.array([[0,0,1],[0,1,1],[2,2,1]]); ok=pred==gt; show(ok,'correct pixels')
assert abs(ok.mean()-8/9)<1e-12

In [ ]:
# 3) class IoU compares intersection to union
pred=np.array([[0,0,1],[0,0,1],[2,2,1]]); gt=np.array([[0,0,1],[0,1,1],[2,2,1]]); c=1; inter=((pred==c)&(gt==c)).sum(); union=((pred==c)|(gt==c)).sum(); plt.figure(figsize=(4,3)); plt.bar(['inter','union'],[inter,union]); plt.title('class IoU')
assert inter==3 and union==4

In [ ]:
# 4) U-Net skip concatenation joins encoder and decoder channels
enc=np.ones((2,2,3)); dec=2*np.ones((2,2,2)); cat=np.concatenate([enc,dec],axis=2); show(cat[:,:,0]+cat[:,:,3],'skip concat')
assert cat.shape==(2,2,5)

In [ ]:
# 5) instance masks keep object identity
m1=np.zeros((4,4)); m1[:2,:2]=1; m2=np.zeros((4,4)); m2[2:,2:]=1; show(m1+2*m2,'instances')
assert m1.sum()==4 and m2.sum()==4